PREPARACION DEL AMBIENTE

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("storageLocation","staerolineasdeveus01")

In [0]:
storageLocation = dbutils.widgets.get("storageLocation")

EXTERNAL LOCATIONS


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-catalog`
URL 'abfss://catalog-ae@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred_aerolineas_dev`)
COMMENT 'Ubicación externa para las tablas de unit-catalog del data lake de los servicios aeroportuarios';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred_aerolineas_dev` )
COMMENT 'Datos raw de empresa aeroportuaria';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-bronze`
URL 'abfss://bronze@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred_aerolineas_dev` )
COMMENT 'Datos Bronze de empresa aeroportuaria';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-silver`
URL 'abfss://silver@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred_aerolineas_dev` )
COMMENT 'Datos Silver de la empresa aeroportuaria';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-golden`
URL 'abfss://golden@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred_aerolineas_dev` )
COMMENT 'Datos Golden de empresa aeroportuaria'

CATÁLOGO

In [0]:
%sql
DROP CATALOG IF EXISTS aerolineas_dev CASCADE;
     

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS aerolineas_dev
MANAGED LOCATION 'abfss://catalog-ae@${storageLocation}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura del ambiente de dev';

ESQUEMAS

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageLocation}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageLocation}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageLocation}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aerolineas_dev.raw;
CREATE SCHEMA IF NOT EXISTS aerolineas_dev.bronze;
CREATE SCHEMA IF NOT EXISTS aerolineas_dev.silver;
CREATE SCHEMA IF NOT EXISTS aerolineas_dev.golden;

## TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_dim_aerolineas (
    aerolinea_id STRING,
    nombre_aerolinea STRING,
    pais_sede STRING,
    hub_iata STRING,
    modelo_negocio STRING,
    fecha_inicio_operaciones DATE,
    activo INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_dim_aerolineas';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_dim_aeropuertos (
    aeropuerto_iata STRING,
    nombre_aeropuerto STRING,
    ciudad STRING,
    pais STRING,
    codigo_pais STRING,
    zona_horaria STRING,
    altitud_m INT,
    latitud DECIMAL(10,6),
    longitud DECIMAL(10,6),
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_dim_aeropuertos';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_dim_aeronaves (
    aeronave_id STRING,
    aerolinea_id STRING,
    matricula STRING,
    modelo STRING,
    categoria STRING,
    capacidad_total INT,
    asientos_business INT,
    asientos_economy INT,
    tipo_combustible STRING,
    anio_fabricacion INT,
    estado_aeronave STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_dim_aeronaves';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_dim_rutas (
    ruta_id STRING,
    origen_iata STRING,
    destino_iata STRING,
    distancia_km INT,
    duracion_programada_min INT,
    tipo_ruta STRING,
    activa INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_dim_rutas';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_fact_vuelos (
    vuelo_id STRING,
    aerolinea_id STRING,
    aeronave_id STRING,
    ruta_id STRING,
    numero_vuelo STRING,
    fecha_salida_programada_utc TIMESTAMP,
    fecha_salida_real_utc TIMESTAMP,
    fecha_llegada_programada_utc TIMESTAMP,
    fecha_llegada_real_utc TIMESTAMP,
    estado_vuelo STRING,
    motivo_cancelacion STRING,
    terminal_salida STRING,
    puerta_embarque STRING,
    demora_salida_min INT,
    demora_llegada_min INT,
    fuente_sistema STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_fact_vuelos';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_fact_pasajeros (
    vuelo_id STRING,
    adultos INT,
    ninos INT,
    infantes_sin_asiento INT,
    pasajeros_con_asiento INT,
    pasajeros_total_a_bordo INT,
    asientos_disponibles INT,
    pasajeros_business INT,
    pasajeros_economy INT,
    transito_conexion INT,
    pasajeros_no_show INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_fact_pasajeros';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_fact_combustible (
    vuelo_id STRING,
    tipo_combustible STRING,
    combustible_inicial_l DECIMAL(18,2),
    combustible_cargado_l DECIMAL(18,2),
    combustible_final_l DECIMAL(18,2),
    combustible_consumido_l DECIMAL(18,2),
    precio_usd_l DECIMAL(10,3),
    costo_carga_usd DECIMAL(18,2),
    proveedor STRING,
    moneda STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_fact_combustible';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_fact_ventas_boletos (
    venta_id STRING,
    vuelo_id STRING,
    fecha_venta_utc TIMESTAMP,
    clase_tarifa STRING,
    canal_venta STRING,
    boletos_vendidos INT,
    boletos_volados INT,
    boletos_no_show INT,
    tarifa_promedio_usd DECIMAL(18,2),
    ingreso_bruto_usd DECIMAL(18,2),
    descuentos_usd DECIMAL(18,2),
    impuestos_usd DECIMAL(18,2),
    ingreso_neto_usd DECIMAL(18,2),
    moneda STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_fact_ventas_boletos';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.bronze.bbdd_fact_equipajes (
    vuelo_id STRING,
    piezas_facturadas INT,
    peso_facturado_kg DECIMAL(18,2),
    piezas_cabina_estimadas INT,
    piezas_extraviadas INT,
    piezas_danadas INT,
    ingreso_equipaje_usd DECIMAL(18,2),
    tiempo_entrega_promedio_min INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/bbdd_fact_equipajes';

## TABLAS SILVER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_dim_aerolineas (
    aerolinea_id STRING,
    nombre_aerolinea STRING,
    pais_sede STRING,
    hub_iata STRING,
    modelo_negocio STRING,
    fecha_inicio_operaciones DATE,
    activo INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_dim_aerolineas';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_dim_aeropuertos (
    aeropuerto_iata STRING,
    nombre_aeropuerto STRING,
    ciudad STRING,
    pais STRING,
    codigo_pais STRING,
    zona_horaria STRING,
    altitud_m INT,
    latitud DECIMAL(10,6),
    longitud DECIMAL(10,6),
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_dim_aeropuertos';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_dim_aeronaves (
    aeronave_id STRING,
    aerolinea_id STRING,
    matricula STRING,
    modelo STRING,
    categoria STRING,
    capacidad_total INT,
    asientos_business INT,
    asientos_economy INT,
    tipo_combustible STRING,
    anio_fabricacion INT,
    estado_aeronave STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_dim_aeronaves';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_dim_rutas (
    ruta_id STRING,
    origen_iata STRING,
    destino_iata STRING,
    distancia_km INT,
    duracion_programada_min INT,
    tipo_ruta STRING,
    activa INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_dim_rutas';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_fact_vuelos (
    vuelo_id STRING,
    aerolinea_id STRING,
    aeronave_id STRING,
    ruta_id STRING,
    numero_vuelo STRING,
    fecha_salida_programada_utc TIMESTAMP,
    fecha_salida_real_utc TIMESTAMP,
    fecha_llegada_programada_utc TIMESTAMP,
    fecha_llegada_real_utc TIMESTAMP,
    estado_vuelo STRING,
    motivo_cancelacion STRING,
    terminal_salida STRING,
    puerta_embarque STRING,
    demora_salida_min INT,
    demora_llegada_min INT,
    fuente_sistema STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_fact_vuelos';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_fact_pasajeros (
    vuelo_id STRING,
    adultos INT,
    ninos INT,
    infantes_sin_asiento INT,
    pasajeros_con_asiento INT,
    pasajeros_total_a_bordo INT,
    asientos_disponibles INT,
    pasajeros_business INT,
    pasajeros_economy INT,
    transito_conexion INT,
    pasajeros_no_show INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_fact_pasajeros';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_fact_combustible (
    vuelo_id STRING,
    tipo_combustible STRING,
    combustible_inicial_l DECIMAL(18,2),
    combustible_cargado_l DECIMAL(18,2),
    combustible_final_l DECIMAL(18,2),
    combustible_consumido_l DECIMAL(18,2),
    precio_usd_l DECIMAL(10,3),
    costo_carga_usd DECIMAL(18,2),
    proveedor STRING,
    moneda STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_fact_combustible';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_fact_ventas_boletos (
    venta_id STRING,
    vuelo_id STRING,
    fecha_venta_utc TIMESTAMP,
    clase_tarifa STRING,
    canal_venta STRING,
    boletos_vendidos INT,
    boletos_volados INT,
    boletos_no_show INT,
    tarifa_promedio_usd DECIMAL(18,2),
    ingreso_bruto_usd DECIMAL(18,2),
    descuentos_usd DECIMAL(18,2),
    impuestos_usd DECIMAL(18,2),
    ingreso_neto_usd DECIMAL(18,2),
    moneda STRING,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_fact_ventas_boletos';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.silver.tbl_fact_equipajes (
    vuelo_id STRING,
    piezas_facturadas INT,
    peso_facturado_kg DECIMAL(18,2),
    piezas_cabina_estimadas INT,
    piezas_extraviadas INT,
    piezas_danadas INT,
    ingreso_equipaje_usd DECIMAL(18,2),
    tiempo_entrega_promedio_min INT,
    fecha_ingesta TIMESTAMP,
    archivo_origen STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/tbl_fact_equipajes';

TABLAS GOLD

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.golden.kpi_aerolinea_mes (
    periodo DATE,
    anio INT,
    mes INT,
    aerolinea_id STRING,
    nombre_aerolinea STRING,
    pais_sede STRING,
    vuelos_programados BIGINT,
    vuelos_operados BIGINT,
    vuelos_cancelados BIGINT,
    vuelos_puntuales BIGINT,
    tasa_cancelacion_pct DECIMAL(10,2),
    puntualidad_pct DECIMAL(10,2),
    demora_salida_promedio_min DECIMAL(10,2),
    demora_llegada_promedio_min DECIMAL(10,2),
    pasajeros_con_asiento BIGINT,
    pasajeros_total_a_bordo BIGINT,
    asientos_ofertados BIGINT,
    ocupacion_pct DECIMAL(10,2),
    pasajeros_no_show BIGINT,
    combustible_cargado_l DECIMAL(20,2),
    combustible_consumido_l DECIMAL(20,2),
    costo_carga_combustible_usd DECIMAL(20,2),
    combustible_l_por_pasajero DECIMAL(18,4),
    ingreso_neto_boletos_usd DECIMAL(20,2),
    ingreso_equipaje_usd DECIMAL(20,2),
    ingreso_total_usd DECIMAL(20,2),
    ingreso_usd_por_pasajero DECIMAL(18,2),
    piezas_facturadas BIGINT,
    piezas_extraviadas BIGINT,
    piezas_danadas BIGINT,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageLocation}.dfs.core.windows.net/kpi_aerolinea_mes';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aerolineas_dev.golden.kpi_ruta_mes (
    periodo DATE,
    anio INT,
    mes INT,
    ruta_id STRING,
    origen_iata STRING,
    nombre_aeropuerto_origen STRING,
    ciudad_origen STRING,
    pais_origen STRING,
    destino_iata STRING,
    nombre_aeropuerto_destino STRING,
    ciudad_destino STRING,
    pais_destino STRING,
    tipo_ruta STRING,
    distancia_km INT,
    aerolineas_operadoras BIGINT,
    vuelos_programados BIGINT,
    vuelos_operados BIGINT,
    vuelos_cancelados BIGINT,
    vuelos_puntuales BIGINT,
    tasa_cancelacion_pct DECIMAL(10,2),
    puntualidad_pct DECIMAL(10,2),
    demora_salida_promedio_min DECIMAL(10,2),
    pasajeros_con_asiento BIGINT,
    pasajeros_total_a_bordo BIGINT,
    asientos_ofertados BIGINT,
    ocupacion_pct DECIMAL(10,2),
    distancia_total_operada_km BIGINT,
    combustible_consumido_l DECIMAL(20,2),
    combustible_l_por_km DECIMAL(18,4),
    combustible_l_por_pasajero DECIMAL(18,4),
    costo_carga_combustible_usd DECIMAL(20,2),
    ingreso_neto_boletos_usd DECIMAL(20,2),
    ingreso_equipaje_usd DECIMAL(20,2),
    ingreso_total_usd DECIMAL(20,2),
    ingreso_usd_por_vuelo DECIMAL(18,2),
    piezas_facturadas BIGINT,
    piezas_extraviadas BIGINT,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageLocation}.dfs.core.windows.net/kpi_ruta_mes';